<a href="https://kaggle.com/kernels/welcome?src=https://github.com/NourEldin-Osama/AI-Notebooks/blob/main/UnslothStudio/UnslothStudio-kaggle-ngrok.ipynb" target="_blank"><img src="https://kaggle.com/static/images/open-in-kaggle.svg" alt="Open In Kaggle"/></a>
<a href="https://colab.research.google.com/github/NourEldin-Osama/AI-Notebooks/blob/main/UnslothStudio/UnslothStudio-kaggle-ngrok.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 🔑 Get your ngrok token
1. Open the ngrok authtoken page here: https://dashboard.ngrok.com/get-started/your-authtoken
2. Sign in or create an ngrok account if needed.
3. Copy your authtoken from the dashboard.
4. Save it as `NGROK_AUTHTOKEN` in Kaggle Secrets or Colab Secrets.

**Kaggle:** Open **Add-ons** → **Secrets**, add a new secret named `NGROK_AUTHTOKEN`, paste the token, and enable it for the notebook.

**Colab:** Open the **Secrets** panel from the left sidebar, add a new secret named `NGROK_AUTHTOKEN`, paste the token, and turn on **Notebook access**.

In [ ]:
# @title 🛠️ Step 1: Setup Unsloth Studio & Dependencies
import os
import subprocess
import time

import httpx

print("🚀 Step 1: Setting up environment and installing dependencies...")

# 1. Install ngrok & httpx
print("📦 Installing ngrok and httpx...")
setup_cmds = """
curl -sSL https://ngrok-agent.s3.amazonaws.com/ngrok.asc | sudo tee /etc/apt/trusted.gpg.d/ngrok.asc >/dev/null
echo "deb https://ngrok-agent.s3.amazonaws.com bookworm main" | sudo tee /etc/apt/sources.list.d/ngrok.list >/dev/null
sudo apt-get update -y >/dev/null 2>&1
sudo apt-get install ngrok -y >/dev/null 2>&1
uv pip install -q httpx
"""
subprocess.run(setup_cmds, shell=True)

# 2. Clone Unsloth
print("📥 Cloning Unsloth repository [Branch: main]...")
subprocess.run(
    "rm -rf unsloth && git clone -q --depth 1 --branch main https://github.com/unslothai/unsloth.git", shell=True
)

# 3. Install Unsloth Studio
print("⚙️ Installing Unsloth Studio (Local Mode). This may take a minute...")
subprocess.run("cat install.sh | bash -s -- --local >/dev/null 2>&1", shell=True, cwd="unsloth")

# 4. Environment Fixes
# Prevent Kaggle/Colab from forcing 'inline' graphs on a headless background server.
os.environ["MPLBACKEND"] = "agg"

print("✅ Setup complete! Proceed to the next cell.")

In [ ]:
# @title 🔐 Step 2: Configure ngrok Authentication


def get_ngrok_token() -> str:
    """Detects the environment and retrieves the ngrok auth token securely."""
    print("🔍 Detecting environment to fetch ngrok token...")

    if "KAGGLE_KERNEL_RUN_TYPE" in os.environ or "KAGGLE_URL_BASE" in os.environ:
        print("🟢 Environment: Kaggle")
        from kaggle_secrets import UserSecretsClient

        return UserSecretsClient().get_secret("NGROK_AUTHTOKEN")

    elif "COLAB_RELEASE_TAG" in os.environ or "COLAB_BACKEND_VERSION" in os.environ:
        print("🟢 Environment: Google Colab")
        from google.colab import userdata

        return userdata.get("NGROK_AUTHTOKEN")

    else:
        print("🟢 Environment: Standard Python / Local")
        return os.getenv("NGROK_AUTHTOKEN")


try:
    NGROK_AUTHTOKEN = get_ngrok_token()
    if not NGROK_AUTHTOKEN:
        raise ValueError("Token is empty or not found.")

    print("🔑 ngrok auth token retrieved. Authenticating...")
    subprocess.run(["ngrok", "config", "add-authtoken", NGROK_AUTHTOKEN], check=True, capture_output=True)
    print("✅ ngrok authenticated successfully!")

except Exception as error:
    print("\n❌ Error retrieving ngrok token.")
    print("💡 Hint: Ensure you have added 'NGROK_AUTHTOKEN' to your Kaggle Secrets or Colab Secrets.")
    raise error

In [ ]:
# @title 🌐 Step 3: Launch Unsloth Studio & ngrok Tunnel

# Start the downloaded ngrok in the background to tunnel port 8889 (where Unsloth Studio will run).
print("⏳ Starting ngrok tunnel in the background on port 8889...")
subprocess.Popen(["ngrok", "http", "8889"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Wait for ngrok to initialize
time.sleep(2)

# Fetch the public URL from ngrok's local API using httpx (it runs on port 4040 by default)
try:
    response = httpx.get("http://localhost:4040/api/tunnels")
    response.raise_for_status()
    public_url = response.json()["tunnels"][0]["public_url"]

    print("\n" + "=" * 60)
    print("✅ Ngrok tunnel established successfully!")
    print(f"🚀 OPEN UNSLOTH STUDIO HERE: {public_url}")
    print("=" * 60 + "\n")

except Exception as e:
    print("\n❌ Failed to retrieve ngrok URL.")
    print(f"Error details: {e}")
    print("💡 Hint: Double-check your NGROK_AUTHTOKEN in step 2.\n")

print("🖥️ Spinning up Unsloth Studio web server...")
print("⚠️ (Keep this cell running to use the Studio UI)\n")

# Start Unsloth Studio Web UI on port 8889 in the background
# Pass the modified environment variables so the matplotlib fix takes effect.
unsloth_bin = os.path.expanduser("~/.local/bin/unsloth")
subprocess.Popen([unsloth_bin, "studio", "-H", "0.0.0.0", "-p", "8889"], env=os.environ)